# BioEmu2 on Colab

Runs Microsoft Research's [BioEmu2](https://github.com/microsoft/bioemu2) (public preview, MIT licence) in three modes:

1. **Structure prediction**: one sharp, most-likely structure.
2. **Equilibrium sampling**: an ensemble of independent conformations.
3. **Time propagation**: chains of time-lagged endpoints (1 ns – 1 µs per step), a coarse trajectory.

The code is adapted from the examples in the BioEmu2 repository and turned into a Colab notebook, so that the model can be tried without a local GPU or a local installation. BioEmu2 is installed from that repository and its weights are downloaded from Hugging Face when you run the cells; neither is included here. Portions adapted from microsoft/bioemu2: Copyright (c) Microsoft Corporation, licensed under the MIT License.

**Before you start:** *Runtime → Change runtime type → GPU*. A free **T4** works for small proteins; with Colab Pro an **L4** (about 2 compute units/hour) is a good default, and an **A100** is faster for large complexes.

**Typical times on an L4** (200 denoising steps per sample): ~200 residues ≈ 9 s/sample at batch 10; ~400 residues ≈ 20 s/sample at batch 5. Install ≈ 3 min, model download ≈ 20 s.

**Notes**
- BioEmu2 is research software. Sampling frequencies are *not* calibrated Boltzmann probabilities, and there is no confidence score (no pLDDT/PAE).
- MSAs come from the public ColabFold server unless you upload your own A3M file.
- This notebook installs BioEmu2 from GitHub, pinned to commit `6d1ca74` (3 Nov 2026), with the CUDA 12 build, which matches Colab's GPU drivers.

## 1. Check the GPU

In [ ]:
#@title Check GPU and Python version
import os

# Set before anything touches CUDA: it lets the allocator grow its segments instead of
# fragmenting, which is the difference between fitting and not on a 15 GB card.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, sys
print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 12), "BioEmu2 needs Python 3.12 or newer."
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv"],
                         capture_output=True, text=True, check=True).stdout)
except Exception:
    raise SystemExit("No GPU found. Use Runtime → Change runtime type → GPU, then run this cell again.")


## 2. Install BioEmu2 (takes a few minutes)

The install replaces Colab's torch with the version BioEmu2 pins, so pip will complain that Colab's CUDA 13 packages no longer match. That is expected and harmless here. If the check cell cannot import torch afterwards, restart the session and run it again.


In [ ]:
#@title Install { display-mode: "form" }
#@markdown BioEmu2 pins torch below 2.14, so Colab's own torch is replaced. pip then reports
#@markdown conflicts with the packages Colab ships against CUDA 13 — torchvision, cuml, cudf,
#@markdown cuda-toolkit. BioEmu2 uses none of them, so those lines are noise; the cell below
#@markdown is what decides whether the install is good.
!pip install -q "bioemu2[cuda12] @ git+https://github.com/microsoft/bioemu2@6d1ca744fa8ed285edafae7b98da65205bf00fd3" py3Dmol \
    --extra-index-url https://download.pytorch.org/whl/cu126 2>&1 | tail -n 20
print("\nIf pip listed conflicts with torchvision, cuml, cudf or cuda-toolkit, ignore them:")
print("they are Colab's CUDA 13 packages and BioEmu2 does not use them.")
print("The cu126 build is the one to install here, and it runs on the CUDA 13 driver.")


In [ ]:
#@title Check the install { display-mode: "form" }
import importlib
try:
    import torch, bioemu2
    print("bioemu2 imported | torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(), "| compute capability",
              "%d.%d" % torch.cuda.get_device_capability())
        print("\nThis is the line that matters. If it says CUDA available: True, the install is fine,")
        print("whatever pip said above.")
    else:
        print("\nNo GPU in this session: Runtime -> Change runtime type -> GPU, then run this cell again.")
except Exception as err:
    print("Import failed:", err)
    print("\nThe usual cause is that torch was already loaded before it was replaced.")
    print("Use Runtime -> Restart session, then run this cell again. The install does not have to be repeated.")


## 3. Define the system

The example is a **complex**: HPr kinase with HPr ([1KKL](https://www.rcsb.org/structure/1KKL)), three copies of the kinase plus one HPr, 614 residues. The kinase makes the HPr site as an oligomer, which is why the copies are there. A single protein is the case BioEmu2's own [examples](https://github.com/microsoft/bioemu2/tree/main/examples) already cover, so this notebook spends its example on the harder one.

To scan something of your own, set `example` to *none* and give either a **PDB ID** (chains and ligands from the first biological assembly) or **sequences** separated by `:`. A ligand can be added by its CCD code or a SMILES string. To drop expression tags or disordered termini, paste trimmed sequences rather than using the PDB ID.


In [ ]:
#@title Input { display-mode: "form" }
example = "1KKL: HPr kinase + HPr"  #@param ["1KKL: HPr kinase + HPr", "none: use the fields below"]
input_type = "pdb_id"  #@param ["pdb_id", "sequence"]
pdb_id = ""  #@param {type:"string"}
include_pdb_ligands = False  #@param {type:"boolean"}
sequence = ""  #@param {type:"string"}
ligand_ccd_code = ""  #@param {type:"string"}
ligand_smiles = ""  #@param {type:"string"}
use_own_msa = False  #@param {type:"boolean"}
job_name = "bioemu2_run"  #@param {type:"string"}

import string
from pathlib import Path
from bioemu2 import System, Protein, Ligand, get_msa

out_root = Path("/content") / job_name
out_root.mkdir(parents=True, exist_ok=True)

EXAMPLE_1KKL = [
    "ERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNEE",
    "ERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNE",
    "AERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNEE",
    "QKTFKVTADSGIHARPATVLVQTASKYDADVNLEYNGKTVNLKSIMGVMSLGIAKGAEITISASGADENDALNALEETMKSERLGE"
]

if example.startswith("1KKL"):          # three kinase copies and one HPr, the crystal constructs
    ids = iter(string.ascii_uppercase)
    entities = [Protein(sequence=s, asym_ids=[next(ids)]) for s in EXAMPLE_1KKL]
    system = System(entities=entities)
elif input_type == "pdb_id":
    system = System.from_pdb_id(pdb_id.strip(), include_ligands=include_pdb_ligands)
    entities = list(system.entities)
else:
    chains = [s.strip().upper().replace(" ", "") for s in sequence.split(":") if s.strip()]
    assert chains, "Enter at least one protein sequence."
    ids = iter(string.ascii_uppercase)
    entities = [Protein(sequence=s, asym_ids=[next(ids)]) for s in chains]
    if ligand_ccd_code.strip():
        entities.append(Ligand(ccd_code=ligand_ccd_code.strip().upper(), asym_ids=[next(ids)]))
    if ligand_smiles.strip():
        entities.append(Ligand(smiles=ligand_smiles.strip(), asym_ids=[next(ids)]))
    system = System(entities=entities)

print(system)

if use_own_msa:
    from google.colab import files
    assert sum(isinstance(e, Protein) for e in entities) == 1, "Own-MSA upload here supports one protein chain only."
    uploaded = files.upload()
    a3m_path = Path("/content") / next(iter(uploaded))
    msa = get_msa(system, source=a3m_path)
else:
    msa = get_msa(system)  # ColabFold server, cached after the first call
print("MSA ready.")

## 4. Load the model

In [ ]:
#@title Load BioEmu2 weights (downloaded from Hugging Face on first use)
import torch
from bioemu2 import load_model

# bfloat16 needs an Ampere-or-newer GPU (A100, L4, …); a T4 falls back to full float32.
major, _ = torch.cuda.get_device_capability()
precision = "bf16-mixed" if major >= 8 else "32-true"
model = load_model(precision=precision)
print("Model loaded with precision", precision, "on", torch.cuda.get_device_name())

## 5. Structure prediction

In [ ]:
#@title Predict the structure
num_structures = 1  #@param {type:"integer"}
seed = 0  #@param {type:"integer"}

from bioemu2 import EmuSession, save_frames, check_physicality
from bioemu2.visualization import view_structure

emu_structure = EmuSession(system, model=model, mode="structure", msa=msa)
prediction = emu_structure.sample(num_samples=num_structures, seed=seed)
save_frames(prediction, out_root / "structure")
print("Saved to", out_root / "structure")
view_structure(prediction, frame=0).show()

del emu_structure                             # its activations are no longer needed
torch.cuda.empty_cache()
print(f"GPU memory held after this step: {torch.cuda.memory_reserved() / 2**30:.1f} GiB")

## 6. Equilibrium sampling
Independent samples from the learned ensemble. Memory is the limit here, not patience: the 614-residue example wants an **L4 or A100**, and on a 15 GB T4 it will fall back to one sample at a time and may not fit at all. A single ~200-residue protein is comfortable anywhere. If the cell reports out-of-memory even at batch 1, restart the session before trying anything else, because a failed attempt leaves memory held.


In [ ]:
#@title Sample the ensemble { display-mode: "form" }
num_samples = 20  #@param {type:"integer"}
batch_size = 0  #@param {type:"integer"}
seed = 1  #@param {type:"integer"}
physicality_mode = "high_quality"  #@param ["high_quality", "high_quality_per_residue", "no_violations"]
#@markdown Leave `batch_size` at 0 to size it from the system (about 2000 residues' worth at a
#@markdown time, as in BioEmu2's own batch runs). If the GPU runs out of memory the cell halves
#@markdown the batch and tries again, down to one sample at a time.
import torch

length = sum(len(e.sequence) for e in system.entities if hasattr(e, "sequence"))
bs = batch_size if batch_size > 0 else max(1, 2000 // max(1, length))

emu_eq = EmuSession(system, model=model, mode="equilibrium", msa=msa)
while True:
    try:
        print(f"sampling {num_samples} of {length} residues at batch {bs} ...", flush=True)
        samples = emu_eq.sample(num_samples=num_samples, batch_size=bs, seed=seed)
        break
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if bs == 1:
            raise RuntimeError(
                "Out of memory even one sample at a time. This GPU is too small for a system of "
                f"{length} residues: use Runtime -> Change runtime type for an L4 or A100, or give "
                "a smaller system in step 3. Restart the session before trying again, since a "
                "failed run leaves memory held.") from None
        bs = max(1, bs // 2)
        print(f"out of memory, retrying at batch {bs}", flush=True)

report = check_physicality(samples, physicality_mode, system=system)
save_frames(samples, out_root / "equilibrium", physicality=report)
n_ok = int(report.passed.sum())
print(f"{n_ok}/{samples.n_frames} samples pass the physicality check" + ("" if report.checked else " (not checked: no protein atoms)"))
print("Saved to", out_root / "equilibrium")


In [ ]:
#@title Plot RMSD to the predicted structure and radius of gyration
import numpy as np, mdtraj as md, matplotlib.pyplot as plt

ca = samples.topology.select("name CA")
rmsd_A = 10 * md.rmsd(samples, prediction, frame=0, atom_indices=ca)
rg_A = 10 * md.compute_rg(samples.atom_slice(ca))
ok = report.passed.astype(bool)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(rmsd_A[ok], bins=30, color="#3b6fb6"); ax[0].set_xlabel("Cα RMSD to predicted structure (Å)"); ax[0].set_ylabel("samples")
ax[1].hist(rg_A[ok], bins=30, color="#3b6fb6"); ax[1].set_xlabel("Cα radius of gyration (Å)")
ax[2].scatter(rmsd_A[ok], rg_A[ok], s=12, color="#3b6fb6", label="pass")
ax[2].scatter(rmsd_A[~ok], rg_A[~ok], s=12, color="#c2410c", label="fail physicality")
ax[2].set_xlabel("Cα RMSD (Å)"); ax[2].set_ylabel("Cα Rg (Å)"); ax[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
#@title View the ensemble (first frames, physically valid samples only)
from bioemu2.visualization import view_trajectory
max_frames = 20  #@param {type:"integer"}
view_trajectory(samples[np.where(ok)[0]] if ok.any() else samples, max_frames=max_frames).show()

## 7. Time propagation
Starts from some equilibrium samples and repeatedly propagates each by `lag_time_ns`. On an L4, one step for 4 starts of a ~200-residue protein takes about 35 s.
Each step is an independent endpoint at that time lag, not an MD frame in between. The documented lag range is 1–1000 ns.

In [ ]:
#@title Simulate time-lagged trajectories
num_starts = 4  #@param {type:"integer"}
lag_time_ns = 10.0  #@param {type:"number"}
num_steps = 10  #@param {type:"integer"}
seed = 2  #@param {type:"integer"}

start_idx = np.where(ok)[0][:num_starts] if ok.any() else np.arange(min(num_starts, samples.n_frames))
x = samples[start_idx]
steps = [x]
for i in range(num_steps):
    x = emu_eq.propagate(x, lag_time_ns=lag_time_ns, seed=seed + i, progress=False)
    steps.append(x)
    print(f"step {i + 1}/{num_steps} done ({(i + 1) * lag_time_ns:g} ns)", flush=True)

trajs = [md.join([s[j] for s in steps]) for j in range(len(start_idx))]
for j, t in enumerate(trajs):
    save_frames(t, out_root / "propagation" / f"start_{j}")
print("Saved to", out_root / "propagation")

time_ns = np.arange(num_steps + 1) * lag_time_ns
plt.figure(figsize=(7, 4))
for j, t in enumerate(trajs):
    plt.plot(time_ns, 10 * md.rmsd(t, prediction, frame=0, atom_indices=ca), marker="o", ms=3, label=f"start {j}")
plt.xlabel("time (ns)"); plt.ylabel("Cα RMSD to predicted structure (Å)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
#@title View one propagated trajectory
trajectory_to_view = 0  #@param {type:"integer"}
view_trajectory(trajs[trajectory_to_view], max_frames=num_steps + 1).show()

## 8. Is the interface consistent?

An ensemble can have contact in every sample and still tell you nothing, if it puts the chains together somewhere different each time. The cell below measures that on the samples from step 6: how many distinct residue pairs are ever touched, how often the commonest pair appears, and how many pairs survive in half the samples. Scattered contact over hundreds of pairs means the interface was not determined, however physical each sample looks.

Run the same measurement on the molecular dynamics of this interface and you get 21/21 frames in contact, 94 distinct pairs, a highest frequency of 1.00 and 14 pairs in half the frames: that is what a determined interface looks like. For the example, a 50-sample BioEmu2 run gave 40/50 samples in contact, 257 distinct pairs, a highest frequency of 0.52 and 2 pairs in half the samples, with 7 samples within fnat > 0.3 of the crystal structure. For comparison, a complex BioEmu2 does not resolve (PKR kinase with eIF2alpha, 2A1A, submitted without the PKR dimer) gave 49/50 samples in contact but 1744 distinct pairs, a highest frequency of 0.12, and none in half.

The second cell puts that ensemble next to molecular dynamics of the same complex, taken from **DynaBench**: three 100 ns replicas for each of the 200-odd complexes of Docking Benchmark 5.5. The notebook carries 21 frames of the 1KKL trajectories, seven from each replica with the first 20 ns of each discarded; the full set is at [http://www-lbt.ibpc.fr/DynaBench](http://www-lbt.ibpc.fr/DynaBench) on the MDposit platform (INRIA node) of the Molecular Dynamics Data Bank. If you use them, cite Barlas AB, Laurent B, Karaca E, Prévost C, Sacquin-Mora S. DynaBench: Dynamic data for the docking benchmark. *Journal of Molecular Biology* 438 (2026) 169650, [doi:10.1016/j.jmb.2026.169650](https://doi.org/10.1016/j.jmb.2026.169650).


In [ ]:
#@title Measure how consistent the interface is { display-mode: "form" }
first_chain = 2 #@param {type:"integer"}
second_chain = 3 #@param {type:"integer"}
contact_cutoff = 4.5 #@param {type:"number"}
#@markdown The two chains to compare, counted from 0. For the example these are the third
#@markdown kinase copy and HPr, the pair the crystal structure puts together.
import numpy as np

if "samples" not in dir():
    print("Run step 6 first: this measures the ensemble sampled there.")
else:
    top = samples.topology
    chains = list(top.chains)
    if max(first_chain, second_chain) >= len(chains):
        raise ValueError(f"this system has {len(chains)} chains, numbered 0 to {len(chains) - 1}")
    ok = np.where(report.passed)[0] if report.passed.any() else np.arange(samples.n_frames)

    def _onehot(ci):
        atoms = top.select(f"chainid {chains[ci].index} and element != H")
        res = np.array([top.atom(i).residue.index for i in atoms])
        res -= res.min()
        hot = np.zeros((len(atoms), res.max() + 1), dtype=np.float32)
        hot[np.arange(len(atoms)), res] = 1.0
        return atoms, hot

    sel_a, hot_a = _onehot(first_chain)
    sel_b, hot_b = _onehot(second_chain)
    freq = np.zeros((hot_a.shape[1], hot_b.shape[1]))
    touching = 0
    for f in ok:
        xa = samples.xyz[f][sel_a] * 10.0                      # A
        xb = samples.xyz[f][sel_b] * 10.0
        close = (np.linalg.norm(xa[:, None, :] - xb[None, :, :], axis=-1) <= contact_cutoff).astype(np.float32)
        pairs = (hot_a.T @ close @ hot_b) > 0
        freq += pairs
        touching += int(pairs.any())
    freq /= len(ok)

    print(f"chain {first_chain} ({hot_a.shape[1]} residues) against chain {second_chain} "
          f"({hot_b.shape[1]} residues), over {len(ok)} physical samples")
    print(f"  samples with any contact          : {touching}/{len(ok)}")
    print(f"  distinct residue pairs touched    : {int((freq > 0).sum())}")
    print(f"  highest frequency of any one pair : {freq.max():.2f}")
    print(f"  pairs in at least half the samples: {int((freq >= 0.5).sum())}")
    print("\n  the same measurement, for scale:")
    print("  the DynaBench MD of this interface : 21/21 in contact, 94 pairs, 1.00, 14 pairs in half.")
    print("  a 50-sample BioEmu2 run of it      : 40/50 in contact, 257 pairs, 0.52, 2 pairs in half.")
    print("  a complex it does not resolve      : 49/50 in contact, 1744 pairs, 0.12, none in half.")
    print("  with few samples the frequencies are coarse: what matters is whether contact concentrates.")


In [ ]:
#@title Look at the ensemble next to the DynaBench MD { display-mode: "form" }
max_frames_shown = 8 #@param {type:"integer"}
#@markdown The left viewer is the BioEmu2 ensemble from step 6, the right one 21 frames of the
#@markdown 1KKL trajectories of the [DynaBench](https://doi.org/10.1016/j.jmb.2026.169650)
#@markdown dataset, seven from each 100 ns replica with the first 20 ns of each discarded.
#@markdown The two chains measured above are coloured, the rest of the system is left pale, and
#@markdown each ensemble is superposed on the first chain of that pair. So the coloured partner
#@markdown moving about is the interface being undecided, not the whole complex drifting.
import urllib.request
from pathlib import Path

import mdtraj as md
import py3Dmol

MD_BASE = "https://raw.githubusercontent.com/CSB-KaracaLab/bioemu2-colab/main/md_reference/"
BACKBONE = "name N or name CA or name C or name O"
PALE, FIRST, SECOND = "lightgrey", "steelblue", "darkorange"
PALE_ALPHA, PAIR_ALPHA = 0.15, 0.7


def _show_ensemble(view, traj, cell, limit, pair):
    """Backbone ribbons of the first few frames, the measured pair picked out."""
    letters = [chr(65 + i) for i in range(len(list(traj.topology.chains)))]
    thin = traj.atom_slice(traj.topology.select(BACKBONE))
    for i in range(min(limit, thin.n_frames)):
        thin[i].save_pdb("/tmp/_frame.pdb")
        view.addModel(Path("/tmp/_frame.pdb").read_text(), "pdb", viewer=cell)
    view.setStyle({}, {"cartoon": {"color": PALE, "opacity": PALE_ALPHA}}, viewer=cell)
    for which, colour in zip(pair, (FIRST, SECOND)):
        if which < len(letters):
            view.setStyle({"chain": letters[which]},
                          {"cartoon": {"color": colour, "opacity": PAIR_ALPHA}}, viewer=cell)
    shown = [letters[w] for w in pair if w < len(letters)]
    view.zoomTo({"chain": shown} if shown else {}, viewer=cell)


def _align_on(traj, chain_index):
    """Superpose on one chain, so the partner's spread is what you see."""
    chains = list(traj.topology.chains)
    if chain_index >= len(chains):
        chain_index = 0
    anchor = traj.topology.select(f"chainid {chains[chain_index].index} and name CA")
    traj.superpose(traj, 0, atom_indices=anchor)
    return traj


def _spread(t, chain_index):
    chains = list(t.topology.chains)
    sel = t.topology.select(f"chainid {chains[min(chain_index, len(chains) - 1)].index} and name CA")
    return 10 * md.rmsd(t, t, 0, atom_indices=sel).max()


if "samples" not in dir():
    print("Run step 6 first.")
else:
    pair = (first_chain, second_chain) if "first_chain" in dir() else (0, 1)
    ok = np.where(report.passed)[0] if report.passed.any() else np.arange(samples.n_frames)
    emu = _align_on(samples[ok], pair[0])

    for name in ("1KKL_md_topology.pdb", "1KKL_md_dynabench.xtc"):
        if not Path(name).is_file():
            urllib.request.urlretrieve(MD_BASE + name, name)
    md_ref = _align_on(md.load("1KKL_md_dynabench.xtc", top="1KKL_md_topology.pdb"), pair[0])

    shape_emu = [c.n_residues for c in emu.topology.chains]
    shape_md = [c.n_residues for c in md_ref.topology.chains]
    print(f"chains {pair[0]} ({FIRST}) and {pair[1]} ({SECOND}) are coloured, aligned on chain {pair[0]}")
    print(f"BioEmu2 : {emu.n_frames} physical samples, chains {shape_emu}, "
          f"chain {pair[1]} moves up to {_spread(emu, pair[1]):.1f} A across the ensemble")
    print(f"MD      : {md_ref.n_frames} frames, chains {shape_md}, "
          f"chain {pair[1]} moves up to {_spread(md_ref, pair[1]):.1f} A")
    if shape_emu != shape_md:
        print("\nDifferent systems: the MD panel is there for scale only, it is always 1KKL.")

    view = py3Dmol.view(width=900, height=420, viewergrid=(1, 2), linked=False)
    _show_ensemble(view, emu, (0, 0), max_frames_shown, pair)
    _show_ensemble(view, md_ref, (0, 1), max_frames_shown, pair)
    view.show()
    print("\nleft: BioEmu2 | right: DynaBench MD")


## 9. Download results

In [ ]:
#@title Zip and download everything
import shutil
from google.colab import files
archive = shutil.make_archive(str(out_root), "zip", out_root)
files.download(archive)

---
Each output folder has `topology.cif` + `samples.xtc` (open in PyMOL, ChimeraX or VMD, or load with MDTraj). `equilibrium/physicality.json` lists which samples failed the checks and why.

Sources: [microsoft/bioemu2](https://github.com/microsoft/bioemu2) README and docs; [model card](https://huggingface.co/microsoft/bioemu2).

The MD reference in the test run comes from the DynaBench dataset: Barlas AB, Laurent B, Karaca E, Prévost C, Sacquin-Mora S. DynaBench: Dynamic data for the docking benchmark. *Journal of Molecular Biology* 438 (2026) 169650, [doi:10.1016/j.jmb.2026.169650](https://doi.org/10.1016/j.jmb.2026.169650). The trajectories are distributed at [http://www-lbt.ibpc.fr/DynaBench](http://www-lbt.ibpc.fr/DynaBench); please cite the paper if you use them. DynaBench is also the name of the Karaca lab's MD analysis pipeline, [CSB-KaracaLab/DynaBench](https://github.com/CSB-KaracaLab/DynaBench), which is a different thing.
